In [3]:
import cv2
import numpy as np

cap = cv2.VideoCapture("road.mp4")
# Use 0 for webcam:
# cap = cv2.VideoCapture(0)

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frame = cv2.resize(frame, (800, 600))

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blur = cv2.GaussianBlur(gray, (5,5), 0)

    edges = cv2.Canny(blur, 50, 150)

    height, width = edges.shape

    mask = np.zeros_like(edges)

    polygon = np.array([[
        (100, height),
        (width-100, height),
        (width//2, height//2)
    ]], np.int32)

    cv2.fillPoly(mask, polygon, 255)

    cropped = cv2.bitwise_and(edges, mask)

    lines = cv2.HoughLinesP(
        cropped,
        2,
        np.pi/180,
        100,
        minLineLength=40,
        maxLineGap=50
    )

    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(frame,
                     (x1, y1),
                     (x2, y2),
                     (0, 255, 0),
                     5)

    cv2.imshow("Lane Detection", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()